# 09 - Mixed Precision Validation

Notebook 08 designs a mixed-precision policy. This notebook validates concrete GGUF candidates produced from that policy under one llama.cpp-based harness.

The central question is no longer "can we build a mixed model?" The question is whether a named candidate keeps enough of the Q4_K_M deployment payoff while recovering enough quality to justify the extra higher-precision tensors.


## 1. Validation Goal

This notebook compares GGUF references and one or more named mixed-precision candidates:

- FP16 GGUF reference
- Q8_0 GGUF
- Q4_K_M GGUF
- Mixed GGUF candidates derived from Notebook 08 by default, with optional experiment overrides

Primary outputs:

- Full HumanEval pass@1 and passed/total counts
- Per-problem generation latency, TTFT, generated tokens, execution time, and total problem time
- Optional short performance benchmark using fixed prompts
- Artifact size and process RSS snapshots
- One validation artifact per iteration at `results/mixed_precision/runs/<iteration_id>.validation.json`
- Optional aggregate pointer at `results/09_mixed_precision_validation.json`


## 2. Methodology Notes

The strongest quality comparison is GGUF-to-GGUF in the same harness. Notebook 07's PyTorch baseline remains useful context, but it is a different backend and should not be treated as the primary deployment reference.

Each iteration writes its own validation file. The old aggregate file can be imported for matching reference runs, but it is not the source of truth for the current iteration. Mixed candidates also get stable model keys based on policy or iteration ids, so a later experiment cannot silently overwrite a generic `mixed` slot.


## 3. Setup


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from IPython.display import Markdown, display
import copy
import gc
import hashlib
import json
import os
import statistics
import sys
import time

import matplotlib.pyplot as plt

try:
    import psutil
except ImportError:
    psutil = None

os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [ ]:
CWD = Path.cwd().resolve()
if (CWD / "results").exists() and (CWD / "notebooks").exists():
    PROJECT_ROOT = CWD
else:
    PROJECT_ROOT = (CWD / "..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "results"
EXPORTS_DIR = PROJECT_ROOT / "exports"
MIXED_RESULTS_DIR = RESULTS_DIR / "mixed_precision"
MIXED_RUNS_DIR = MIXED_RESULTS_DIR / "runs"
MIXED_TENSOR_TYPES_DIR = MIXED_RESULTS_DIR / "tensor_types"

for directory in [RESULTS_DIR, MIXED_RESULTS_DIR, MIXED_RUNS_DIR, MIXED_TENSOR_TYPES_DIR]:
    directory.mkdir(exist_ok=True)

PTQ05_PATH = RESULTS_DIR / "05_ptq_artifacts_summary.json"
PERF06_PATH = RESULTS_DIR / "06_benchmarking_perf_snapshot.json"
BASELINE07_PATH = RESULTS_DIR / "07_full_baseline_humaneval.json"
POLICY08_PATH = RESULTS_DIR / "08_mixed_precision_policy.json"
VALIDATION09_AGGREGATE_PATH = RESULTS_DIR / "09_mixed_precision_validation.json"
ITER_LEDGER_JSONL_PATH = RESULTS_DIR / "09_iteration_ledger.jsonl"

MODEL_ID = "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct"
DATASET_ID = "openai/openai_humaneval"
FULL_HUMANEVAL_PROBLEMS = 164


def env_flag(name: str, default: bool) -> bool:
    raw = os.environ.get(name)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "on"}


In [ ]:
# Heavy work is opt-in. Flip these here or set environment variables before execution.
RUN_FULL_HUMANEVAL = env_flag("RUN_FULL_HUMANEVAL", False)
RUN_FRESH_PERF_BENCHMARKS = env_flag("RUN_FRESH_PERF_BENCHMARKS", False)
REEVALUATE_EXISTING = env_flag("REEVALUATE_EXISTING", False)
IMPORT_LEGACY_09_AGGREGATE = env_flag("IMPORT_LEGACY_09_AGGREGATE", True)
WRITE_AGGREGATE_VALIDATION = env_flag("WRITE_AGGREGATE_VALIDATION", False)
APPEND_ITERATION_LEDGER = env_flag("APPEND_ITERATION_LEDGER", False)
HASH_GGUF_ARTIFACTS = env_flag("HASH_GGUF_ARTIFACTS", False)

# None means "use Notebook 08's recommended policy id".
ACTIVE_ITERATION_ID_OVERRIDE = None
ITERATION_NOTES = "Notebook 08 recommended candidate."

# Use this list for explicit experiments. By default, Notebook 09 consumes Notebook 08 directly.
MIXED_CANDIDATE_OVERRIDES = []

# None means references plus every configured mixed candidate for HumanEval, and mixed candidates for perf.
HUMANEVAL_MODEL_KEYS_OVERRIDE = None
PERF_MODEL_KEYS_OVERRIDE = None

GGUF_N_CTX = 2048
GGUF_N_THREADS = 8
GGUF_N_GPU_LAYERS = -1
HUMANEVAL_MAX_PROBLEMS = 164
GENERATION_MAX_TOKENS = 512
GENERATION_TEMPERATURE = 0.0
EXEC_TIMEOUT_SEC = 10

print(f"Project root: {PROJECT_ROOT}")
print(f"Per-iteration runs: {MIXED_RUNS_DIR}")
print(f"Run full HumanEval: {RUN_FULL_HUMANEVAL}")
print(f"Run fresh perf benchmarks: {RUN_FRESH_PERF_BENCHMARKS}")
print(f"Reevaluate existing runs: {REEVALUATE_EXISTING}")
print(f"Import matching legacy aggregate runs: {IMPORT_LEGACY_09_AGGREGATE}")
print(f"Write optional aggregate: {WRITE_AGGREGATE_VALIDATION}")
print(f"Append iteration ledger: {APPEND_ITERATION_LEDGER}")
print(f"Hash GGUF artifacts: {HASH_GGUF_ARTIFACTS}")


In [ ]:
def load_json(path: Path, required: bool = True):
    if path.exists():
        with path.open() as f:
            return json.load(f)
    if required:
        raise FileNotFoundError(f"Missing required artifact: {path}")
    return None


def save_json(path: Path, payload) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w") as f:
        json.dump(payload, f, indent=2)
    print(f"Saved {path}")


def markdown_table(rows, columns):
    if not rows:
        return "_No rows to display._"

    header = "| " + " | ".join(label for _, label in columns) + " |"
    divider = "| " + " | ".join("---" for _ in columns) + " |"
    body = []

    for row in rows:
        values = []
        for key, _ in columns:
            value = row.get(key, "")
            if value is None:
                value = ""
            elif isinstance(value, float):
                value = f"{value:.2f}"
            elif isinstance(value, list):
                value = ", ".join(str(v) for v in value)
            elif isinstance(value, bool):
                value = "yes" if value else "no"
            values.append(str(value))
        body.append("| " + " | ".join(values) + " |")

    return "\n".join([header, divider, *body])


def safe_mean(values):
    clean = [value for value in values if value is not None]
    return statistics.mean(clean) if clean else None


def safe_stdev(values):
    clean = [value for value in values if value is not None]
    return statistics.pstdev(clean) if len(clean) > 1 else 0.0 if clean else None


def safe_div(numerator, denominator):
    if numerator is None or denominator in (None, 0):
        return None
    return numerator / denominator


def delta(a, b):
    if a is None or b is None:
        return None
    return a - b


In [ ]:
def process_rss_gib():
    if psutil is None:
        return None
    return psutil.Process().memory_info().rss / (1024 ** 3)


def path_size_gib(path):
    if path is None:
        return None
    path = Path(path)
    if not path.exists():
        return None
    return path.stat().st_size / (1024 ** 3)


def sha256_file(path):
    if path is None:
        return None
    path = Path(path)
    if not path.exists():
        return None
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def optional_sha256_file(path, enabled: bool):
    if path is None:
        return None, "missing"
    path = Path(path)
    if not path.exists():
        return None, "missing"
    if not enabled:
        return None, "skipped_disabled"
    return sha256_file(path), "computed"


def read_jsonl(path: Path):
    if not path.exists():
        return []
    rows = []
    with path.open() as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def append_jsonl(path: Path, row):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a") as f:
        f.write(json.dumps(row) + "\n")


def stable_digest(payload) -> str:
    encoded = json.dumps(payload, sort_keys=True, separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


## 4. Load Notebook 08 Policy

This is the key cleanup from the earlier scratchpad version. The default mixed candidate is not hardcoded here. Notebook 09 reads Notebook 08's policy file, uses its recommended policy id as the candidate key, and uses the tensor-type and artifact paths recorded by Notebook 08.


In [ ]:
ptq05 = load_json(PTQ05_PATH)
perf06 = load_json(PERF06_PATH, required=False)
baseline07 = load_json(BASELINE07_PATH, required=False)
policy08 = load_json(POLICY08_PATH)
legacy09_aggregate = load_json(VALIDATION09_AGGREGATE_PATH, required=False)

artifacts05 = ptq05["artifacts"]
recommended_policy_id = policy08["recommendation"]["recommended_policy_id"]
recommended_policy = next(
    policy for policy in policy08["candidate_policies"]
    if policy["policy_id"] == recommended_policy_id
)
mixed_plan08 = policy08["mixed_artifact_plan"]
tensor_manifest08 = policy08["tensor_type_manifest"]

ACTIVE_ITERATION_ID = ACTIVE_ITERATION_ID_OVERRIDE or recommended_policy_id
VALIDATION_RUN_PATH = MIXED_RUNS_DIR / f"{ACTIVE_ITERATION_ID}.validation.json"
TENSOR_TYPES_SNAPSHOT_PATH = MIXED_TENSOR_TYPES_DIR / f"{ACTIVE_ITERATION_ID}.tensor_types.txt"

print(f"Notebook 08 recommended policy: {recommended_policy_id}")
print(f"Active iteration id: {ACTIVE_ITERATION_ID}")
print(f"Validation run file: {VALIDATION_RUN_PATH}")
print(f"Tensor-type source: {tensor_manifest08['path']}")
print(f"Mixed artifact source: {mixed_plan08['path']}")


In [ ]:
def normalize_precision(value):
    if value is None:
        return None
    return str(value).strip().lower().replace("-", "_")


def mixed_candidate_from_policy08(policy, policy_payload):
    mixed_plan = policy_payload["mixed_artifact_plan"]
    tensor_manifest = policy_payload["tensor_type_manifest"]
    protected_layers = policy.get("protected_layers") or policy_payload["recommendation"]["protected_layers"]
    default_precision = normalize_precision(policy.get("default_precision"))
    protected_precision = normalize_precision(mixed_plan.get("protected_tensor_precision", "q8_0"))
    policy_id = policy["policy_id"]

    return {
        "model_key": policy_id,
        "iteration_id": policy_id,
        "policy_id": policy_id,
        "label": f"{policy['label']} mixed ({','.join(str(layer) for layer in protected_layers)})",
        "precision": f"{default_precision}_default_{protected_precision}_protected",
        "artifact_path": mixed_plan["path"],
        "artifact_role": "Notebook 08 recommended candidate",
        "is_mixed_candidate": True,
        "default_precision": default_precision,
        "protected_precision": protected_precision,
        "protected_layers": protected_layers,
        "lower_precision_layers": policy.get("lower_precision_layers", []),
        "tensor_types_path": tensor_manifest["path"],
        "source_policy_path": str(POLICY08_PATH),
        "notes": ITERATION_NOTES,
    }


MIXED_CANDIDATES = [mixed_candidate_from_policy08(recommended_policy, policy08)]
MIXED_CANDIDATES.extend(copy.deepcopy(MIXED_CANDIDATE_OVERRIDES))

for candidate in MIXED_CANDIDATES:
    candidate.setdefault("model_key", candidate["iteration_id"])
    candidate.setdefault("policy_id", candidate["iteration_id"])
    candidate.setdefault("is_mixed_candidate", True)
    candidate.setdefault("artifact_role", "mixed precision experiment")
    candidate.setdefault("source_policy_path", str(POLICY08_PATH))
    candidate.setdefault("notes", "manual Notebook 09 override")

MIXED_CANDIDATE_KEYS = [candidate["model_key"] for candidate in MIXED_CANDIDATES]
REFERENCE_MODEL_KEYS = ["fp16", "q8_0", "q4_k_m"]
HUMANEVAL_MODEL_KEYS = HUMANEVAL_MODEL_KEYS_OVERRIDE or [*REFERENCE_MODEL_KEYS, *MIXED_CANDIDATE_KEYS]
PERF_MODEL_KEYS = PERF_MODEL_KEYS_OVERRIDE or [*MIXED_CANDIDATE_KEYS]

print("Mixed candidates:")
for candidate in MIXED_CANDIDATES:
    print(f"- {candidate['model_key']}: {candidate['artifact_path']}")
print(f"HumanEval model keys: {HUMANEVAL_MODEL_KEYS}")
print(f"Performance model keys: {PERF_MODEL_KEYS}")


## 5. Build Model Manifest

The manifest now has stable reference entries plus any number of mixed candidate entries. There is no special mutable `mixed` key.


In [ ]:
MODEL_MANIFEST = {
    "fp16": {
        "label": "FP16 GGUF",
        "precision": "fp16",
        "artifact_path": artifacts05["f16_source"]["path"],
        "artifact_role": "GGUF FP16 reference",
        "is_mixed_candidate": False,
    },
    "q8_0": {
        "label": "Q8_0 GGUF",
        "precision": "q8_0",
        "artifact_path": artifacts05["q8_0"]["path"],
        "artifact_role": "conservative quantized baseline",
        "is_mixed_candidate": False,
    },
    "q4_k_m": {
        "label": "Q4_K_M GGUF",
        "precision": "q4_k_m",
        "artifact_path": artifacts05["q4_k_m"]["path"],
        "artifact_role": "aggressive quantized baseline",
        "is_mixed_candidate": False,
    },
}

for candidate in MIXED_CANDIDATES:
    MODEL_MANIFEST[candidate["model_key"]] = candidate

for model_key, cfg in MODEL_MANIFEST.items():
    artifact_path = Path(cfg["artifact_path"])
    cfg["artifact_exists"] = artifact_path.exists()
    cfg["artifact_size_gib"] = path_size_gib(artifact_path)
    tensor_types_path = cfg.get("tensor_types_path")
    if tensor_types_path:
        cfg["tensor_types_exists"] = Path(tensor_types_path).exists()
        cfg["tensor_types_sha256"] = sha256_file(tensor_types_path)
    else:
        cfg["tensor_types_exists"] = None
        cfg["tensor_types_sha256"] = None

manifest_rows = [
    {
        "model_key": model_key,
        "label": cfg["label"],
        "exists": cfg["artifact_exists"],
        "size_gib": cfg["artifact_size_gib"],
        "tensor_types_exists": cfg.get("tensor_types_exists"),
        "role": cfg["artifact_role"],
    }
    for model_key, cfg in MODEL_MANIFEST.items()
]

display(Markdown(markdown_table(manifest_rows, [
    ("model_key", "Key"),
    ("label", "Model"),
    ("exists", "Artifact"),
    ("size_gib", "Size GiB"),
    ("tensor_types_exists", "Tensor file"),
    ("role", "Role"),
])))


In [ ]:
def new_validation_run():
    now = datetime.now(timezone.utc).isoformat()
    return {
        "timestamp_utc": now,
        "updated_at_utc": now,
        "notebook": "09_mixed_precision_validation",
        "iteration_id": ACTIVE_ITERATION_ID,
        "model_id": MODEL_ID,
        "dataset": DATASET_ID,
        "source_policy_path": str(POLICY08_PATH),
        "run_artifact_path": str(VALIDATION_RUN_PATH),
        "config": {},
        "model_manifest": {},
        "humaneval_runs": {},
        "performance_runs": {},
        "performance_snapshots": [],
        "comparison": {},
        "notes": [],
    }


validation_run = load_json(VALIDATION_RUN_PATH, required=False) or new_validation_run()
validation_run["updated_at_utc"] = datetime.now(timezone.utc).isoformat()
validation_run["iteration_id"] = ACTIVE_ITERATION_ID
validation_run["source_policy_path"] = str(POLICY08_PATH)
validation_run["run_artifact_path"] = str(VALIDATION_RUN_PATH)
validation_run["config"] = {
    "run_full_humaneval": RUN_FULL_HUMANEVAL,
    "run_fresh_perf_benchmarks": RUN_FRESH_PERF_BENCHMARKS,
    "reevaluate_existing": REEVALUATE_EXISTING,
    "import_legacy_09_aggregate": IMPORT_LEGACY_09_AGGREGATE,
    "write_aggregate_validation": WRITE_AGGREGATE_VALIDATION,
    "humaneval_model_keys": HUMANEVAL_MODEL_KEYS,
    "perf_model_keys": PERF_MODEL_KEYS,
    "humaneval_max_problems": HUMANEVAL_MAX_PROBLEMS,
    "full_humaneval_problems": FULL_HUMANEVAL_PROBLEMS,
    "gguf_n_ctx": GGUF_N_CTX,
    "gguf_n_threads": GGUF_N_THREADS,
    "gguf_n_gpu_layers": GGUF_N_GPU_LAYERS,
    "generation_max_tokens": GENERATION_MAX_TOKENS,
    "generation_temperature": GENERATION_TEMPERATURE,
    "exec_timeout_sec": EXEC_TIMEOUT_SEC,
}
validation_run["model_manifest"] = copy.deepcopy(MODEL_MANIFEST)


## 6. Context From Prior Notebooks


In [ ]:
context_rows = []
if baseline07:
    ev = baseline07["evaluation"]
    context_rows.append({
        "source": "Notebook 07 PyTorch baseline",
        "pass_at_1": ev["pass_at_1_percent"],
        "passed": f"{ev['n_passed']}/{ev['n_total']}",
        "note": "context only; different backend from GGUF",
    })

if policy08:
    dry = policy08["mixed_artifact_plan"].get("dry_run", {})
    build = policy08["mixed_artifact_plan"].get("build", {})
    verify = policy08["mixed_artifact_plan"].get("verification", {})
    context_rows.append({
        "source": "Notebook 08 recommended policy",
        "pass_at_1": None,
        "passed": "not measured in Notebook 08",
        "note": (
            f"policy={recommended_policy_id}; build={build.get('status')}; "
            f"verification={verify.get('status')}; dry-run size={dry.get('quant_size_gib'):.2f} GiB"
            if dry.get("quant_size_gib") else f"policy={recommended_policy_id}; build={build.get('status')}"
        ),
    })

display(Markdown(markdown_table(context_rows, [
    ("source", "Source"),
    ("pass_at_1", "Pass@1"),
    ("passed", "Passed"),
    ("note", "Note"),
])))


## 7. HumanEval GGUF Harness


In [ ]:
from datasets import load_dataset
from llama_cpp import Llama
from utils.humaneval_helpers import clean_and_extract, execute_with_timeout

HUMANEVAL = None


def get_humaneval():
    global HUMANEVAL
    if HUMANEVAL is None:
        HUMANEVAL = load_dataset(DATASET_ID, split="test")
    return HUMANEVAL


def humaneval_subset(max_problems: int):
    dataset = get_humaneval()
    n = min(max_problems, len(dataset))
    return dataset.select(range(n))


def format_humaneval_instruction(problem):
    return (
        "Complete the following Python function. Return raw Python code only with correct "
        "newlines and indentation. Do not use markdown fences or explanations. "
        "You may return either the full function or just the function body.\n\n"
        f"{problem['prompt']}"
    )


def format_humaneval_messages(problem):
    return [{"role": "user", "content": format_humaneval_instruction(problem)}]


def chunk_text_from_chat_stream(chunk):
    choice = chunk.get("choices", [{}])[0]
    delta = choice.get("delta") or {}
    if isinstance(delta, dict):
        content = delta.get("content")
        if content:
            return content
    message = choice.get("message") or {}
    if isinstance(message, dict):
        content = message.get("content")
        if content:
            return content
    return choice.get("text") or ""


def count_tokens(llm, text: str):
    if not text:
        return 0
    return len(llm.tokenize(text.encode("utf-8"), add_bos=False))


def load_llama_model(model_key, cfg):
    path = Path(cfg["artifact_path"])
    if not path.exists():
        raise FileNotFoundError(f"Missing GGUF artifact for {model_key}: {path}")
    return Llama(
        model_path=str(path),
        n_ctx=GGUF_N_CTX,
        n_threads=GGUF_N_THREADS,
        n_gpu_layers=GGUF_N_GPU_LAYERS,
        verbose=False,
    )


### Stream A GGUF Completion

This cell isolates the generation step: prompt in, streamed text and token timing out.


In [ ]:
def generate_gguf_completion_streamed(problem, llm, max_tokens=GENERATION_MAX_TOKENS):
    messages = format_humaneval_messages(problem)
    started = time.perf_counter()
    first_token_time = None
    chunks = []

    stream = llm.create_chat_completion(
        messages=messages,
        max_tokens=max_tokens,
        temperature=GENERATION_TEMPERATURE,
        stream=True,
    )

    for chunk in stream:
        piece = chunk_text_from_chat_stream(chunk)
        if piece and first_token_time is None:
            first_token_time = time.perf_counter()
        chunks.append(piece)

    ended = time.perf_counter()
    generated = "".join(chunks).replace("\r\n", "\n").replace("\r", "\n").rstrip()
    generated_tokens = count_tokens(llm, generated)
    ttft_sec = first_token_time - started if first_token_time else None
    generation_latency_sec = ended - started
    decode_tokens_per_sec = (
        generated_tokens / (ended - first_token_time)
        if first_token_time is not None and ended > first_token_time and generated_tokens > 0
        else None
    )

    return generated, {
        "ttft_sec": ttft_sec,
        "generation_latency_sec": generation_latency_sec,
        "generated_tokens": generated_tokens,
        "decode_tokens_per_sec": decode_tokens_per_sec,
    }


### Evaluate One HumanEval Problem

This is where generation becomes a benchmark result: clean the completion, execute the test, and record per-problem timing.


In [ ]:
def evaluate_gguf_humaneval_problem(problem, llm):
    problem_started = time.perf_counter()
    generated, generation_metrics = generate_gguf_completion_streamed(problem, llm)

    clean_started = time.perf_counter()
    code = clean_and_extract(generated, problem)
    clean_latency_sec = time.perf_counter() - clean_started

    full_code = code + "\n\n" + problem["test"] + f"\ncheck({problem['entry_point']})"
    execution_started = time.perf_counter()
    result = execute_with_timeout(full_code, timeout=EXEC_TIMEOUT_SEC)
    execution_latency_sec = time.perf_counter() - execution_started

    return {
        "task_id": problem["task_id"],
        "entry_point": problem["entry_point"],
        "result": result,
        "passed": result == "passed",
        "generated": generated,
        "cleaned_code_preview": code[:500],
        "ttft_sec": generation_metrics["ttft_sec"],
        "generation_latency_sec": generation_metrics["generation_latency_sec"],
        "generated_tokens": generation_metrics["generated_tokens"],
        "decode_tokens_per_sec": generation_metrics["decode_tokens_per_sec"],
        "clean_latency_sec": clean_latency_sec,
        "execution_latency_sec": execution_latency_sec,
        "problem_total_sec": time.perf_counter() - problem_started,
    }


### Summarize A HumanEval Run

This cell collapses per-problem rows into the pass@1 and latency metrics used later.


In [ ]:
def summarize_humaneval_results(results, runtime_sec_total):
    n_total = len(results)
    n_passed = sum(1 for row in results if row["passed"])
    return {
        "n_total": n_total,
        "n_passed": n_passed,
        "pass_at_1_percent": 100.0 * n_passed / n_total if n_total else 0.0,
        "runtime_sec_total": runtime_sec_total,
        "runtime_min_total": runtime_sec_total / 60,
        "ttft_sec_mean": safe_mean([row["ttft_sec"] for row in results]),
        "ttft_sec_std": safe_stdev([row["ttft_sec"] for row in results]),
        "generation_latency_sec_mean": safe_mean([row["generation_latency_sec"] for row in results]),
        "generation_latency_sec_std": safe_stdev([row["generation_latency_sec"] for row in results]),
        "problem_total_sec_mean": safe_mean([row["problem_total_sec"] for row in results]),
        "problem_total_sec_std": safe_stdev([row["problem_total_sec"] for row in results]),
        "execution_latency_sec_mean": safe_mean([row["execution_latency_sec"] for row in results]),
        "generated_tokens_mean": safe_mean([row["generated_tokens"] for row in results]),
        "decode_tokens_per_sec_mean": safe_mean([row["decode_tokens_per_sec"] for row in results]),
    }


## 8. Run Or Load Full HumanEval

A run is considered complete only when it covers all 164 HumanEval problems. Imported reference runs are labeled as imports so the comparison table does not hide where a number came from.

Before running with `RUN_FULL_HUMANEVAL=True`, make a concrete prediction: should the active mixed candidate land closer to Q8_0 or Q4_K_M, and by how many pass@1 points?


In [ ]:
def same_path(a, b):
    if not a or not b:
        return False
    return Path(a).expanduser().resolve() == Path(b).expanduser().resolve()


def mark_humaneval_source(run, model_key, cfg, source, imported_from=None):
    run = copy.deepcopy(run)
    run["model_key"] = model_key
    run["label"] = cfg["label"]
    run["artifact_path"] = cfg["artifact_path"]
    run["artifact_size_gib"] = cfg.get("artifact_size_gib")
    run["source"] = source
    if imported_from:
        run["imported_from"] = str(imported_from)
    return run


def import_matching_legacy_humaneval(model_key, cfg):
    if not IMPORT_LEGACY_09_AGGREGATE or not legacy09_aggregate:
        return None

    legacy_runs = legacy09_aggregate.get("humaneval_runs", {})
    candidate_keys = [model_key, *cfg.get("legacy_model_keys", [])]
    for legacy_key in candidate_keys:
        legacy_run = legacy_runs.get(legacy_key)
        if legacy_run and same_path(legacy_run.get("artifact_path"), cfg.get("artifact_path")):
            return mark_humaneval_source(
                legacy_run,
                model_key,
                cfg,
                source="imported_legacy_09_aggregate",
                imported_from=VALIDATION09_AGGREGATE_PATH,
            )
    return None


def get_or_import_humaneval_run(model_key, cfg):
    existing = validation_run.get("humaneval_runs", {}).get(model_key)
    if existing:
        existing.setdefault("source", "run_artifact")
        return existing

    imported = import_matching_legacy_humaneval(model_key, cfg)
    if imported:
        validation_run.setdefault("humaneval_runs", {})[model_key] = imported
        print(f"Imported matching HumanEval run for {model_key} from legacy aggregate.")
        return imported

    return None


In [ ]:
def run_humaneval_for_model(model_key, cfg):
    print(f"\n=== HumanEval: {model_key} ({cfg['label']}) ===")
    problems = humaneval_subset(HUMANEVAL_MAX_PROBLEMS)
    started = time.perf_counter()
    llm = load_llama_model(model_key, cfg)
    rss_after_load = process_rss_gib()
    results = []

    try:
        for i, problem in enumerate(problems, start=1):
            result = evaluate_gguf_humaneval_problem(problem, llm)
            results.append(result)
            n_passed = sum(1 for item in results if item["passed"])
            running_pass = 100.0 * n_passed / i
            print(
                f"{model_key} {result['task_id']}: {result['result']} "
                f"({result['problem_total_sec']:.1f}s) | running pass@1: {running_pass:.2f}%"
            )
    finally:
        del llm
        gc.collect()

    runtime_sec_total = time.perf_counter() - started
    summary = summarize_humaneval_results(results, runtime_sec_total)
    return {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "model_key": model_key,
        "label": cfg["label"],
        "artifact_path": cfg["artifact_path"],
        "artifact_size_gib": cfg.get("artifact_size_gib"),
        "source": "fresh_09_humaneval",
        "dataset": DATASET_ID,
        "generation_config": {
            "max_tokens": GENERATION_MAX_TOKENS,
            "temperature": GENERATION_TEMPERATURE,
            "n_ctx": GGUF_N_CTX,
            "n_threads": GGUF_N_THREADS,
            "n_gpu_layers": GGUF_N_GPU_LAYERS,
        },
        "memory": {
            "process_rss_gib_after_load": rss_after_load,
            "process_rss_gib_after_cleanup": process_rss_gib(),
            "note": "Process RSS snapshots; not isolated model memory.",
        },
        "summary": summary,
        "results": results,
    }


In [ ]:
for model_key in HUMANEVAL_MODEL_KEYS:
    cfg = MODEL_MANIFEST[model_key]
    existing = get_or_import_humaneval_run(model_key, cfg)

    if existing and not REEVALUATE_EXISTING:
        print(f"Skipping {model_key}: existing/imported HumanEval run is available.")
        continue

    if not RUN_FULL_HUMANEVAL:
        print(f"Skipping {model_key}: RUN_FULL_HUMANEVAL is False")
        continue

    run = run_humaneval_for_model(model_key, cfg)
    validation_run.setdefault("humaneval_runs", {})[model_key] = run
    validation_run["updated_at_utc"] = datetime.now(timezone.utc).isoformat()
    save_json(VALIDATION_RUN_PATH, validation_run)


## 9. Optional Fresh Performance Benchmark

Before running with `RUN_FRESH_PERF_BENCHMARKS=True`, predict whether the protected Q8_0 tensors will make the mixed candidate throughput closer to Q4_K_M or Q8_0.


In [ ]:
BENCH_PROMPTS = [
    "Write a Python function `is_prime(n:int) -> bool` with type hints and docstring.",
    "Implement `merge_intervals(intervals)` in Python and explain edge cases briefly.",
    "Given a list of integers, return the longest increasing subsequence length in Python.",
]
BENCH_ROUNDS = 2
BENCH_MAX_TOKENS = 128
EXPECTED_PERF_SAMPLES = len(BENCH_PROMPTS) * BENCH_ROUNDS


def run_perf_trial(llm, prompt, max_tokens=BENCH_MAX_TOKENS):
    started = time.perf_counter()
    first_token_time = None
    chunks = []
    stream = llm.create_chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=GENERATION_TEMPERATURE,
        stream=True,
    )
    for chunk in stream:
        piece = chunk_text_from_chat_stream(chunk)
        if piece and first_token_time is None:
            first_token_time = time.perf_counter()
        chunks.append(piece)

    ended = time.perf_counter()
    generated = "".join(chunks)
    generated_tokens = count_tokens(llm, generated)
    ttft_sec = first_token_time - started if first_token_time else None
    total_latency_sec = ended - started
    decode_tokens_per_sec = (
        generated_tokens / (ended - first_token_time)
        if first_token_time is not None and ended > first_token_time and generated_tokens > 0
        else None
    )
    return {
        "ttft_sec": ttft_sec,
        "total_latency_sec": total_latency_sec,
        "decode_tokens_per_sec": decode_tokens_per_sec,
        "generated_tokens": generated_tokens,
        "generated_preview": generated[:160],
    }


def summarize_perf_trials(trials):
    return {
        "n_samples": len(trials),
        "ttft_sec_mean": safe_mean([trial["ttft_sec"] for trial in trials]),
        "ttft_sec_std": safe_stdev([trial["ttft_sec"] for trial in trials]),
        "tokens_per_sec_mean": safe_mean([trial["decode_tokens_per_sec"] for trial in trials]),
        "tokens_per_sec_std": safe_stdev([trial["decode_tokens_per_sec"] for trial in trials]),
        "total_latency_sec_mean": safe_mean([trial["total_latency_sec"] for trial in trials]),
        "total_latency_sec_std": safe_stdev([trial["total_latency_sec"] for trial in trials]),
        "generated_tokens_mean": safe_mean([trial["generated_tokens"] for trial in trials]),
    }


In [ ]:
def mark_perf_source(run, model_key, cfg, source, imported_from=None):
    run = copy.deepcopy(run)
    run["model_key"] = model_key
    run["label"] = cfg["label"]
    run["artifact_path"] = cfg["artifact_path"]
    run["artifact_size_gib"] = cfg.get("artifact_size_gib")
    run["source"] = source
    if imported_from:
        run["imported_from"] = str(imported_from)
    return run


def import_matching_legacy_perf(model_key, cfg):
    if not IMPORT_LEGACY_09_AGGREGATE or not legacy09_aggregate:
        return None

    legacy_runs = legacy09_aggregate.get("performance_runs", {})
    candidate_keys = [model_key, *cfg.get("legacy_model_keys", [])]
    for legacy_key in candidate_keys:
        legacy_run = legacy_runs.get(legacy_key)
        if legacy_run and same_path(legacy_run.get("artifact_path"), cfg.get("artifact_path")):
            return mark_perf_source(
                legacy_run,
                model_key,
                cfg,
                source="imported_legacy_09_aggregate",
                imported_from=VALIDATION09_AGGREGATE_PATH,
            )
    return None


def get_or_import_perf_run(model_key, cfg):
    existing = validation_run.get("performance_runs", {}).get(model_key)
    if existing:
        existing.setdefault("source", "run_artifact")
        return existing

    imported = import_matching_legacy_perf(model_key, cfg)
    if imported:
        validation_run.setdefault("performance_runs", {})[model_key] = imported
        print(f"Imported matching performance run for {model_key} from legacy aggregate.")
        return imported

    return None


In [ ]:
def run_perf_benchmark_for_model(model_key, cfg):
    print(f"\n=== Performance benchmark: {model_key} ({cfg['label']}) ===")
    llm = load_llama_model(model_key, cfg)
    rss_after_load = process_rss_gib()
    trials = []
    started = time.perf_counter()

    try:
        _ = run_perf_trial(llm, BENCH_PROMPTS[0], max_tokens=32)
        for round_idx in range(BENCH_ROUNDS):
            for prompt_idx, prompt in enumerate(BENCH_PROMPTS):
                trial = run_perf_trial(llm, prompt)
                trial["round"] = round_idx + 1
                trial["prompt_index"] = prompt_idx
                trials.append(trial)
                print(
                    f"{model_key} trial {len(trials)}: "
                    f"ttft={trial['ttft_sec']:.4f}s, "
                    f"tps={trial['decode_tokens_per_sec']:.2f}, "
                    f"lat={trial['total_latency_sec']:.4f}s"
                )
    finally:
        del llm
        gc.collect()

    return {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "model_key": model_key,
        "label": cfg["label"],
        "artifact_path": cfg["artifact_path"],
        "artifact_size_gib": cfg.get("artifact_size_gib"),
        "source": "fresh_09_perf",
        "runtime_sec_total": time.perf_counter() - started,
        "benchmark_config": {
            "prompts": BENCH_PROMPTS,
            "rounds": BENCH_ROUNDS,
            "max_tokens": BENCH_MAX_TOKENS,
            "temperature": GENERATION_TEMPERATURE,
            "n_ctx": GGUF_N_CTX,
            "n_threads": GGUF_N_THREADS,
            "n_gpu_layers": GGUF_N_GPU_LAYERS,
        },
        "memory": {
            "process_rss_gib_after_load": rss_after_load,
            "process_rss_gib_after_cleanup": process_rss_gib(),
            "note": "Process RSS snapshots; not isolated model memory.",
        },
        "summary": summarize_perf_trials(trials),
        "trials": trials,
    }


In [ ]:
for model_key in PERF_MODEL_KEYS:
    cfg = MODEL_MANIFEST[model_key]
    existing = get_or_import_perf_run(model_key, cfg)

    if existing and not REEVALUATE_EXISTING:
        print(f"Skipping {model_key}: existing/imported performance run is available.")
        continue

    if not RUN_FRESH_PERF_BENCHMARKS:
        print(f"Skipping {model_key}: RUN_FRESH_PERF_BENCHMARKS is False")
        continue

    run = run_perf_benchmark_for_model(model_key, cfg)
    validation_run.setdefault("performance_runs", {})[model_key] = run
    validation_run["updated_at_utc"] = datetime.now(timezone.utc).isoformat()
    save_json(VALIDATION_RUN_PATH, validation_run)


## 10. Imported Performance Snapshots

These rows are useful for orientation, but they are explicitly marked as snapshots. A fresh Notebook 09 performance run is cleaner for final claims because it uses the same active manifest and session.


In [ ]:
def existing_perf_snapshot_rows():
    rows = []

    if perf06:
        benchmark = perf06["benchmark_results"]
        for model_key in REFERENCE_MODEL_KEYS:
            measurement = benchmark["measurements"].get(model_key)
            artifact = benchmark["artifacts"].get(model_key)
            cfg = MODEL_MANIFEST.get(model_key, {})
            if not measurement or not artifact:
                continue
            perf = measurement["performance"]
            memory = measurement.get("memory", {})
            rows.append({
                "model_key": model_key,
                "source": "06_imported_perf_snapshot",
                "artifact_path": cfg.get("artifact_path"),
                "artifact_size_gib": artifact.get("artifact_size_gib"),
                "ttft_sec_mean": perf.get("ttft_sec_mean"),
                "tokens_per_sec_mean": perf.get("tokens_per_sec_mean"),
                "total_latency_sec_mean": perf.get("total_latency_sec_mean"),
                "peak_rss_gib": memory.get("peak_rss_gib"),
                "n_samples": perf.get("n_samples"),
                "freshness": "imported snapshot",
            })

    for model_key, cfg in MODEL_MANIFEST.items():
        if not cfg.get("is_mixed_candidate"):
            continue
        mixed_bench = policy08.get("mixed_artifact_plan", {}).get("benchmark", {})
        mixed_build = policy08.get("mixed_artifact_plan", {}).get("build", {})
        mixed_summary = mixed_bench.get("summary")
        if mixed_summary and same_path(mixed_bench.get("artifact_path"), cfg.get("artifact_path")):
            rows.append({
                "model_key": model_key,
                "source": "08_imported_mixed_perf_snapshot",
                "artifact_path": cfg.get("artifact_path"),
                "artifact_size_gib": mixed_build.get("artifact_size_gib"),
                "ttft_sec_mean": mixed_summary.get("ttft_sec_mean"),
                "tokens_per_sec_mean": mixed_summary.get("tokens_per_sec_mean"),
                "total_latency_sec_mean": mixed_summary.get("total_latency_sec_mean"),
                "peak_rss_gib": None,
                "n_samples": mixed_summary.get("n_samples"),
                "freshness": "imported snapshot",
            })

    return rows


existing_perf_rows = existing_perf_snapshot_rows()
validation_run["performance_snapshots"] = existing_perf_rows

display(Markdown(markdown_table(existing_perf_rows, [
    ("model_key", "Model"),
    ("source", "Source"),
    ("artifact_size_gib", "Size GiB"),
    ("ttft_sec_mean", "TTFT s"),
    ("tokens_per_sec_mean", "Tok/s"),
    ("total_latency_sec_mean", "Latency s"),
    ("peak_rss_gib", "RSS GiB"),
    ("freshness", "Freshness"),
])))


## 11. Comparison Tables


In [ ]:
def humaneval_status(summary):
    if not summary:
        return "missing"
    n_total = summary.get("n_total", 0)
    if n_total >= FULL_HUMANEVAL_PROBLEMS:
        return "complete"
    if n_total > 0:
        return "partial"
    return "missing"


def humaneval_reference():
    fp16_run = validation_run.get("humaneval_runs", {}).get("fp16")
    if fp16_run:
        return {
            "pass_at_1_percent": fp16_run["summary"]["pass_at_1_percent"],
            "label": "GGUF FP16",
            "backend": "llama.cpp/GGUF",
            "source": fp16_run.get("source", "run_artifact"),
        }

    if baseline07:
        ev = baseline07["evaluation"]
        return {
            "pass_at_1_percent": ev["pass_at_1_percent"],
            "label": "Notebook 07 PyTorch baseline",
            "backend": "transformers/PyTorch",
            "source": "context_only",
        }

    return None


quality_reference = humaneval_reference()
print(f"Quality reference: {quality_reference['label'] if quality_reference else 'missing'}")


In [ ]:
def quality_rows_from_validation():
    rows = []
    reference_pass = quality_reference["pass_at_1_percent"] if quality_reference else None
    reference_label = quality_reference["label"] if quality_reference else None

    if baseline07:
        ev = baseline07["evaluation"]
        rows.append({
            "model_key": "pytorch_baseline_07",
            "label": "Notebook 07 PyTorch baseline",
            "backend": "transformers/PyTorch",
            "pass_at_1_percent": ev["pass_at_1_percent"],
            "passed": f"{ev['n_passed']}/{ev['n_total']}",
            "delta_vs_reference": None,
            "baseline_reference_used": "context only",
            "quality_source": "07_full_baseline_humaneval",
            "runtime_min_total": ev["runtime_sec_total"] / 60,
            "problem_total_sec_mean": None,
            "ttft_sec_mean": None,
            "run_scope": f"{ev['n_total']}/{FULL_HUMANEVAL_PROBLEMS}",
            "status": humaneval_status(ev),
        })

    for model_key, cfg in MODEL_MANIFEST.items():
        run = validation_run.get("humaneval_runs", {}).get(model_key)
        if not run:
            rows.append({
                "model_key": model_key,
                "label": cfg["label"],
                "backend": "llama.cpp/GGUF",
                "pass_at_1_percent": None,
                "passed": "not run",
                "delta_vs_reference": None,
                "baseline_reference_used": reference_label,
                "quality_source": "missing",
                "runtime_min_total": None,
                "problem_total_sec_mean": None,
                "ttft_sec_mean": None,
                "run_scope": f"0/{FULL_HUMANEVAL_PROBLEMS}",
                "status": "missing",
            })
            continue

        summary = run["summary"]
        rows.append({
            "model_key": model_key,
            "label": cfg["label"],
            "backend": "llama.cpp/GGUF",
            "pass_at_1_percent": summary["pass_at_1_percent"],
            "passed": f"{summary['n_passed']}/{summary['n_total']}",
            "delta_vs_reference": delta(summary["pass_at_1_percent"], reference_pass),
            "baseline_reference_used": reference_label,
            "quality_source": run.get("source", "run_artifact"),
            "runtime_min_total": summary.get("runtime_min_total"),
            "problem_total_sec_mean": summary.get("problem_total_sec_mean"),
            "ttft_sec_mean": summary.get("ttft_sec_mean"),
            "run_scope": f"{summary.get('n_total', 0)}/{FULL_HUMANEVAL_PROBLEMS}",
            "status": humaneval_status(summary),
        })

    return rows


quality_rows = quality_rows_from_validation()
display(Markdown(markdown_table(quality_rows, [
    ("model_key", "Model"),
    ("backend", "Backend"),
    ("pass_at_1_percent", "Pass@1"),
    ("passed", "Passed"),
    ("delta_vs_reference", "Delta vs ref"),
    ("baseline_reference_used", "Reference"),
    ("quality_source", "Quality source"),
    ("run_scope", "Scope"),
    ("status", "Status"),
])))


In [ ]:
def perf_status_from_samples(n_samples):
    if n_samples is None:
        return "missing"
    if n_samples >= EXPECTED_PERF_SAMPLES:
        return "complete"
    if n_samples > 0:
        return "partial"
    return "missing"


def preferred_perf_by_model():
    preferred = {}

    for row in existing_perf_rows:
        preferred[row["model_key"]] = row

    for model_key, run in validation_run.get("performance_runs", {}).items():
        summary = run["summary"]
        preferred[model_key] = {
            "model_key": model_key,
            "source": run.get("source", "run_artifact"),
            "artifact_size_gib": run.get("artifact_size_gib"),
            "ttft_sec_mean": summary.get("ttft_sec_mean"),
            "tokens_per_sec_mean": summary.get("tokens_per_sec_mean"),
            "total_latency_sec_mean": summary.get("total_latency_sec_mean"),
            "peak_rss_gib": run.get("memory", {}).get("process_rss_gib_after_load"),
            "n_samples": summary.get("n_samples"),
            "freshness": "fresh" if run.get("source") == "fresh_09_perf" else "imported run",
        }

    return preferred


perf_by_model = preferred_perf_by_model()
perf_rows = []
for model_key, cfg in MODEL_MANIFEST.items():
    row = perf_by_model.get(model_key, {})
    n_samples = row.get("n_samples")
    perf_rows.append({
        "model_key": model_key,
        "label": cfg["label"],
        "perf_source": row.get("source", "missing"),
        "perf_freshness": row.get("freshness", "missing"),
        "artifact_size_gib": cfg.get("artifact_size_gib"),
        "ttft_sec_mean": row.get("ttft_sec_mean"),
        "tokens_per_sec_mean": row.get("tokens_per_sec_mean"),
        "total_latency_sec_mean": row.get("total_latency_sec_mean"),
        "peak_rss_gib": row.get("peak_rss_gib"),
        "n_samples": n_samples,
        "perf_status": perf_status_from_samples(n_samples),
    })

fp16_perf = perf_by_model.get("fp16", {})
for row in perf_rows:
    row["speedup_vs_fp16"] = safe_div(row.get("tokens_per_sec_mean"), fp16_perf.get("tokens_per_sec_mean"))
    ratio = safe_div(row.get("total_latency_sec_mean"), fp16_perf.get("total_latency_sec_mean"))
    row["latency_reduction_vs_fp16_percent"] = (1 - ratio) * 100 if ratio is not None else None


display(Markdown(markdown_table(perf_rows, [
    ("model_key", "Model"),
    ("perf_source", "Perf source"),
    ("perf_freshness", "Freshness"),
    ("artifact_size_gib", "Size GiB"),
    ("ttft_sec_mean", "TTFT s"),
    ("tokens_per_sec_mean", "Tok/s"),
    ("speedup_vs_fp16", "Speedup"),
    ("total_latency_sec_mean", "Latency s"),
    ("latency_reduction_vs_fp16_percent", "Latency red %"),
    ("peak_rss_gib", "RSS GiB"),
    ("perf_status", "Status"),
])))


In [ ]:
def combined_rows():
    quality_by_model = {row["model_key"]: row for row in quality_rows}
    perf_by_key = {row["model_key"]: row for row in perf_rows}
    rows = []

    for model_key, cfg in MODEL_MANIFEST.items():
        q = quality_by_model.get(model_key, {})
        p = perf_by_key.get(model_key, {})
        rows.append({
            "model_key": model_key,
            "label": cfg["label"],
            "size_gib": cfg.get("artifact_size_gib"),
            "pass_at_1_percent": q.get("pass_at_1_percent"),
            "passed": q.get("passed"),
            "quality_delta": q.get("delta_vs_reference"),
            "baseline_reference_used": q.get("baseline_reference_used"),
            "quality_source": q.get("quality_source"),
            "quality_status": q.get("status"),
            "perf_source": p.get("perf_source"),
            "perf_freshness": p.get("perf_freshness"),
            "microbench_tps": p.get("tokens_per_sec_mean"),
            "microbench_ttft": p.get("ttft_sec_mean"),
            "microbench_latency": p.get("total_latency_sec_mean"),
            "rss_gib": p.get("peak_rss_gib"),
            "perf_status": p.get("perf_status"),
        })
    return rows


joined_rows = combined_rows()
display(Markdown(markdown_table(joined_rows, [
    ("model_key", "Model"),
    ("size_gib", "Size GiB"),
    ("pass_at_1_percent", "Pass@1"),
    ("quality_delta", "Quality delta"),
    ("baseline_reference_used", "Quality ref"),
    ("quality_source", "Quality source"),
    ("quality_status", "Quality status"),
    ("perf_source", "Perf source"),
    ("perf_freshness", "Perf freshness"),
    ("microbench_tps", "Tok/s"),
    ("perf_status", "Perf status"),
])))


## 12. Visual Comparisons


In [ ]:
plot_rows = [row for row in joined_rows if row.get("pass_at_1_percent") is not None]
if plot_rows:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    labels = [row["model_key"] for row in plot_rows]

    axes[0].bar(labels, [row["size_gib"] for row in plot_rows], color="#4b5563")
    axes[0].set_title("Artifact Size")
    axes[0].set_ylabel("GiB")

    axes[1].bar(labels, [row["pass_at_1_percent"] for row in plot_rows], color="#2563eb")
    axes[1].set_title("HumanEval Pass@1")
    axes[1].set_ylabel("percent")

    axes[2].bar(labels, [row.get("microbench_tps") or 0 for row in plot_rows], color="#059669")
    axes[2].set_title("Microbenchmark Throughput")
    axes[2].set_ylabel("tokens/sec")

    for ax in axes:
        ax.tick_params(axis="x", rotation=25)

    plt.tight_layout()
    plt.show()
else:
    print("No completed GGUF HumanEval runs available yet for plots.")


## 13. Save Validation Artifact


In [ ]:
validation_run["comparison"] = {
    "quality_rows": quality_rows,
    "performance_rows": perf_rows,
    "combined_rows": joined_rows,
    "reference_notes": [
        "GGUF FP16 is the primary quality reference when available.",
        "Notebook 07 PyTorch baseline is context only unless no GGUF FP16 quality run exists.",
        "Performance rows distinguish fresh Notebook 09 runs from imported snapshots.",
        "A HumanEval row is complete only when it covers all 164 problems.",
    ],
}
validation_run["updated_at_utc"] = datetime.now(timezone.utc).isoformat()
save_json(VALIDATION_RUN_PATH, validation_run)

if WRITE_AGGREGATE_VALIDATION:
    aggregate = {
        "updated_at_utc": validation_run["updated_at_utc"],
        "notebook": "09_mixed_precision_validation",
        "latest_iteration_id": ACTIVE_ITERATION_ID,
        "latest_run_artifact_path": str(VALIDATION_RUN_PATH),
        "run_artifacts_dir": str(MIXED_RUNS_DIR),
        "comparison": validation_run["comparison"],
    }
    save_json(VALIDATION09_AGGREGATE_PATH, aggregate)
else:
    print(f"Aggregate not written. Current source of truth is {VALIDATION_RUN_PATH}")


## 14. Iteration Ledger

The ledger is now derived from the per-iteration run artifact. It summarizes candidates, tensor-file hashes, optional GGUF hashes, metric sources, and deltas without making the ledger itself the source of truth.


### Candidate Summary Rows

This gathers quality, performance, artifact, and tensor-manifest metadata for each model key.


In [ ]:
def summary_for_model(model_key):
    quality = next((row for row in quality_rows if row.get("model_key") == model_key), {})
    perf = next((row for row in perf_rows if row.get("model_key") == model_key), {})
    cfg = MODEL_MANIFEST[model_key]
    artifact_sha256, artifact_hash_status = optional_sha256_file(cfg.get("artifact_path"), HASH_GGUF_ARTIFACTS)
    return {
        "model_key": model_key,
        "label": cfg["label"],
        "policy_id": cfg.get("policy_id"),
        "protected_layers": cfg.get("protected_layers"),
        "default_precision": cfg.get("default_precision"),
        "protected_precision": cfg.get("protected_precision"),
        "artifact_path": cfg.get("artifact_path"),
        "artifact_sha256": artifact_sha256,
        "artifact_hash_status": artifact_hash_status,
        "artifact_size_gib": cfg.get("artifact_size_gib"),
        "tensor_types_path": cfg.get("tensor_types_path"),
        "tensor_types_sha256": cfg.get("tensor_types_sha256"),
        "pass_at_1_percent": quality.get("pass_at_1_percent"),
        "passed": quality.get("passed"),
        "quality_delta_vs_reference": quality.get("quality_delta") or quality.get("delta_vs_reference"),
        "quality_source": quality.get("quality_source"),
        "quality_status": quality.get("status"),
        "perf_tps": perf.get("tokens_per_sec_mean"),
        "perf_ttft_sec": perf.get("ttft_sec_mean"),
        "perf_latency_sec": perf.get("total_latency_sec_mean"),
        "perf_source": perf.get("perf_source"),
        "perf_freshness": perf.get("perf_freshness"),
        "perf_status": perf.get("perf_status"),
    }


### Build Ledger Row

The ledger row is derived from the validation artifact and comparison tables; it is a summary, not the source of truth.


In [ ]:
def build_ledger_row():
    candidate_summaries = [summary_for_model(key) for key in MIXED_CANDIDATE_KEYS]
    primary = candidate_summaries[0] if candidate_summaries else None
    references = {key: summary_for_model(key) for key in REFERENCE_MODEL_KEYS}

    row = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "iteration_id": ACTIVE_ITERATION_ID,
        "notes": ITERATION_NOTES,
        "run_artifact_path": str(VALIDATION_RUN_PATH),
        "source_policy_path": str(POLICY08_PATH),
        "reference_models": references,
        "mixed_candidates": candidate_summaries,
        "primary_candidate_key": primary["model_key"] if primary else None,
    }

    q4 = references.get("q4_k_m", {})
    q8 = references.get("q8_0", {})
    fp16 = references.get("fp16", {})
    if primary:
        row["primary_candidate_metrics"] = {
            "pass_at_1_percent": primary.get("pass_at_1_percent"),
            "delta_pass_vs_fp16": delta(primary.get("pass_at_1_percent"), fp16.get("pass_at_1_percent")),
            "delta_pass_vs_q8": delta(primary.get("pass_at_1_percent"), q8.get("pass_at_1_percent")),
            "delta_pass_vs_q4": delta(primary.get("pass_at_1_percent"), q4.get("pass_at_1_percent")),
            "tokens_per_sec": primary.get("perf_tps"),
            "delta_tps_vs_q4": delta(primary.get("perf_tps"), q4.get("perf_tps")),
            "artifact_size_gib": primary.get("artifact_size_gib"),
        }
    else:
        row["primary_candidate_metrics"] = {}

    fingerprint_payload = copy.deepcopy(row)
    fingerprint_payload.pop("timestamp_utc", None)
    row["ledger_fingerprint"] = stable_digest(fingerprint_payload)
    return row


### Preview Ledger Row

Preview the compact row before optionally appending it to the JSONL ledger.


In [ ]:
ledger_row = build_ledger_row()
preview = {
    "iteration_id": ledger_row["iteration_id"],
    "primary_candidate": ledger_row.get("primary_candidate_key"),
    "pass_at_1": ledger_row["primary_candidate_metrics"].get("pass_at_1_percent"),
    "delta_pass_vs_q4": ledger_row["primary_candidate_metrics"].get("delta_pass_vs_q4"),
    "tok_s": ledger_row["primary_candidate_metrics"].get("tokens_per_sec"),
    "delta_tps_vs_q4": ledger_row["primary_candidate_metrics"].get("delta_tps_vs_q4"),
    "artifact_gib": ledger_row["primary_candidate_metrics"].get("artifact_size_gib"),
    "fingerprint": ledger_row["ledger_fingerprint"][:12],
}

display(Markdown(markdown_table([preview], [
    ("iteration_id", "Iteration ID"),
    ("primary_candidate", "Primary candidate"),
    ("pass_at_1", "Pass@1"),
    ("delta_pass_vs_q4", "Delta pass vs Q4"),
    ("tok_s", "Tok/s"),
    ("delta_tps_vs_q4", "Delta tok/s vs Q4"),
    ("artifact_gib", "Artifact GiB"),
    ("fingerprint", "Fingerprint"),
])))


In [ ]:
if APPEND_ITERATION_LEDGER:
    tensor_src = Path(MIXED_CANDIDATES[0].get("tensor_types_path", "")) if MIXED_CANDIDATES else None
    if tensor_src and tensor_src.exists():
        TENSOR_TYPES_SNAPSHOT_PATH.write_text(tensor_src.read_text())
        ledger_row.setdefault("files", {})["tensor_types_snapshot_path"] = str(TENSOR_TYPES_SNAPSHOT_PATH)

    existing_ledger_rows = read_jsonl(ITER_LEDGER_JSONL_PATH)
    seen_fingerprints = {row.get("ledger_fingerprint") for row in existing_ledger_rows}
    if ledger_row["ledger_fingerprint"] in seen_fingerprints:
        print("Ledger row already exists for this exact iteration summary; not appending a duplicate.")
    else:
        append_jsonl(ITER_LEDGER_JSONL_PATH, ledger_row)
        print(f"Appended ledger row to {ITER_LEDGER_JSONL_PATH}")
else:
    print("Ledger not appended. Set APPEND_ITERATION_LEDGER=True to append this row.")


## 15. Iteration History


In [ ]:
ledger_rows = read_jsonl(ITER_LEDGER_JSONL_PATH)

history_rows = []
for row in ledger_rows[-20:]:
    primary = row.get("primary_candidate_metrics", {})
    candidate_key = row.get("primary_candidate_key")

    # Backward compatibility for older ledger rows created before per-run artifacts.
    if not primary and row.get("metrics"):
        primary = {
            "pass_at_1_percent": row.get("metrics", {}).get("humaneval", {}).get("mixed_pass_at_1_percent"),
            "delta_pass_vs_q4": row.get("metrics", {}).get("humaneval", {}).get("delta_pass_vs_q4"),
            "tokens_per_sec": row.get("metrics", {}).get("performance", {}).get("mixed_tps"),
            "artifact_size_gib": row.get("files", {}).get("artifact_size_gib"),
        }
        candidate_key = "legacy_mixed_slot"

    history_rows.append({
        "timestamp_utc": row.get("timestamp_utc"),
        "iteration_id": row.get("iteration_id"),
        "candidate": candidate_key,
        "pass_at_1": primary.get("pass_at_1_percent"),
        "delta_vs_q4": primary.get("delta_pass_vs_q4"),
        "tok_s": primary.get("tokens_per_sec"),
        "artifact_gib": primary.get("artifact_size_gib"),
    })

display(Markdown(markdown_table(history_rows, [
    ("timestamp_utc", "Timestamp UTC"),
    ("iteration_id", "Iteration ID"),
    ("candidate", "Candidate"),
    ("pass_at_1", "Pass@1"),
    ("delta_vs_q4", "Delta pass vs Q4"),
    ("tok_s", "Tok/s"),
    ("artifact_gib", "Artifact GiB"),
])))


## 16. Interpretation


In [ ]:
quality_by_model = {row["model_key"]: row for row in quality_rows}
perf_by_model_rows = {row["model_key"]: row for row in perf_rows}

lines = []
for candidate_key in MIXED_CANDIDATE_KEYS:
    quality = quality_by_model.get(candidate_key, {})
    perf = perf_by_model_rows.get(candidate_key, {})

    if quality.get("status") == "complete":
        lines.append(
            f"{candidate_key} HumanEval pass@1: {quality['pass_at_1_percent']:.2f}% "
            f"({quality['passed']}); reference={quality.get('baseline_reference_used')}; "
            f"source={quality.get('quality_source')}."
        )
    elif quality.get("status") == "partial":
        lines.append(
            f"{candidate_key} has only a partial HumanEval run "
            f"({quality.get('run_scope')}); do not make final quality claims yet."
        )
    else:
        lines.append(f"{candidate_key} HumanEval has not been run yet.")

    if perf.get("tokens_per_sec_mean") is not None:
        lines.append(
            f"{candidate_key} performance: {perf['tokens_per_sec_mean']:.2f} tok/s; "
            f"source={perf.get('perf_source')}; freshness={perf.get('perf_freshness')}."
        )
    else:
        lines.append(f"{candidate_key} has no performance result yet.")

lines.append(
    "Go/no-go criterion: a mixed candidate should materially improve quality over Q4_K_M while retaining most of the Q4_K_M size advantage. "
    "If performance is slower than expected, prioritize full HumanEval quality first, then profile protected Q8_0 tensor overhead."
)

display(Markdown("\n".join(f"- {line}" for line in lines)))
